In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.6 Data quality and eval card

Post-training lives or dies on data quality, a few thousand clean, diverse examples
beat a million noisy ones. And every adapted model should ship an **eval card**:
what it was trained on, how it was measured, against what baseline, and where it
fails.

In [ ]:
import json
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss, perplexity)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=2, n_embd=64)
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(150 if os.environ.get("POCKETLM_CI") else 400):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
ppl = perplexity(estimate_loss(model, data, 32, 16, iters=10, generator=g))
eval_card = {
    "model": "PocketLM (u7 demo)",
    "task": "char-level next-token on Gutenberg #100",
    "metrics": {"perplexity": round(ppl, 2), "uniform_baseline": tok.vocab_size},
    "data": "Project Gutenberg eBook #100, public domain, leading 1MB slice",
    "limitations": ["char-level", "tiny model", "demo-scale post-training only"],
}
print(json.dumps(eval_card, indent=2))

An eval card makes a result honest and reproducible: the number means nothing
without the baseline, the data provenance, and the stated limits beside it.

In [ ]:
assert set(eval_card) >= {"model", "task", "metrics", "data", "limitations"}
assert eval_card["metrics"]["perplexity"] < tok.vocab_size